#### 문서의 내용을 읽고 쪼개기

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
  chunk_size= 1500,       ## 하나의 청크가 가질 토큰 수(청크 크기)
  chunk_overlap= 200      ## 청크 간에 중복시킬 토큰 수
)

loader= Docx2txtLoader('Tax.docx')


document_list  = loader.load_and_split(text_splitter= splitter)

document_list

#### 문서 임베딩 후 벡터 데이터베이스로 저장

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
%pip install langchain-upstage

In [3]:
from langchain_upstage import UpstageEmbeddings

embedding = UpstageEmbeddings(model = 'embedding-passage')

#### ChromaDB로 임베디드 DB 만들기

In [ ]:
%pip install langchain-chroma

In [4]:
from langchain_chroma import Chroma

# 처음 DB 생성할 때는 이 방식으로

# database = Chroma.from_documents(
#   documents= document_list,
#   embedding=embedding, 
#   collection_name='chroma-tax',
#   persist_directory= 'upstage_chroma'
# )

database = Chroma(
  collection_name= 'chroma-tax',
  persist_directory= './upstage_croma',
  embedding_function= embedding     # 요거 안쓰면 Chroma 기본 차원수 384
)

#### Retrieve

In [5]:
query = '기타소득의 세율을 기타소득의 종류별로 설명해주세요'

retrieved_docs = database.similarity_search(query=query, k=20)

In [6]:
retrieved_docs

[]

#### Augmented Generation

In [25]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
  model = 'claude-3-haiku-20240307',
  temperature = 0.3,    # 높을수록 창의적, 랜덤
  top_p = 1             # 낮을수록 확실한 토큰만 선택
)

In [8]:
prompt = f'''[Identity]
- 당신은 최고의 한국 소득세 전문가입니다.
- [Context]를 참고하여 사용자의 [Question]에 답변해주세요.

[Context]
{retrieved_docs}

[Question]
{query}
'''

In [9]:
ai_message= llm.invoke(prompt)

In [10]:
ai_message.content

'네, 기타소득의 세율은 종류에 따라 다음과 같습니다:\n\n1. 경품 및 포상금: \n   - 경품 및 포상금의 경우 22% 세율이 적용됩니다. \n   - 다만 1회 수령금액이 40만원 이하인 경우 비과세 대상이 됩니다.\n\n2. 저작권 사용료:\n   - 저작권 사용료는 16.5% 세율이 적용됩니다.\n   - 다만 연간 2,000만원 이하의 저작권 사용료는 분리과세 대상이 됩니다.\n\n3. 강연료 및 회의참석료:\n   - 강연료 및 회의참석료는 16.5% 세율이 적용됩니다.\n\n4. 보험모집인 수수료:\n   - 보험모집인 수수료는 16.5% 세율이 적용됩니다.\n\n5. 투자조합 배분금:\n   - 투자조합 배분금은 14% 세율이 적용됩니다.\n\n이처럼 기타소득의 종류에 따라 세율이 다르게 적용되므로, 정확한 세금 납부를 위해서는 개별 상황을 고려해야 합니다. 궁금한 점이 더 있으시면 말씀해 주시기 바랍니다.'

#### LCEL

In [ ]:
%pip install -U langchain langchain-community langchain-core

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

# 프롬프트 정의
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 context를 바탕으로 질문에 답하세요.\n\nContext: {context}"),
    ("human", "{question}")
])

# 문서 포맷팅 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 체인 구성
qa_chain = (
    {
        "context": database.as_retriever(k=20) | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 실행
answer = qa_chain.invoke(query)
print(answer)

기타소득의 세율은 소득의 종류에 따라 다음과 같이 적용됩니다:

1. 일시적 보수 및 이와 유사한 성질의 소득:
   - 세율: 20% (지방소득세 포함 시 22%)
   - 예: 강연료, 출연료, 심사료 등

2. 저작권 사용료, 특허권 사용료 등 무체재산권 사용료:
   - 세율: 20% (지방소득세 포함 시 22%)

3. 경품 및 포상금:
   - 세율: 20% (지방소득세 포함 시 22%)

4. 보험차익:
   - 세율: 20% (지방소득세 포함 시 22%)

5. 기타 기타소득:
   - 세율: 14% (지방소득세 포함 시 15.4%)
   - 예: 경조금, 사례금, 임대료 등

이와 같이 기타소득의 종류에 따라 세율이 다르게 적용됩니다. 일반적으로 20%(지방소득세 포함 시 22%)의 세율이 가장 많이 적용되며, 그 외 특별한 경우에는 14%(지방소득세 포함 시 15.4%)의 세율이 적용됩니다.


In [12]:
query = '사업소득이 있는 자가 받을 수 있는 세액공제에 대해서 알려주세요'
answer = qa_chain.invoke(query)
print(answer)

사업소득이 있는 자가 받을 수 있는 주요 세액공제는 다음과 같습니다:

1. 근로소득세액공제
- 사업소득이 있는 자도 근로소득이 있다면 근로소득세액공제를 받을 수 있습니다.
- 근로소득이 7천만원 이하인 경우 최대 130만원까지 공제받을 수 있습니다.

2. 연금보험료 세액공제
- 국민연금, 퇴직연금 등 연금보험료를 납부한 경우 그 금액의 12%를 세액공제받을 수 있습니다.
- 연간 최대 120만원까지 공제받을 수 있습니다.

3. 특별세액공제
- 의료비, 교육비, 기부금 등에 대해 특별세액공제를 받을 수 있습니다.
- 공제한도와 공제율은 항목별로 다릅니다.

4. 중소기업 사업주 세액공제
- 중소기업 사업주의 경우 인건비 등 일정 비용에 대해 추가 세액공제를 받을 수 있습니다.

이 외에도 사업용 자산 투자 세액공제, 가업 승계 세액공제 등 다양한 세액공제 제도가 있으니 개인의 상황에 맞춰 활용하시기 바랍니다.


In [13]:
query = '2. 전자계산서 발급 세액공제(제56조의3)의 요건에 대해서 알려주세요'
answer = qa_chain.invoke(query)
print(answer)

전자계산서 발급 세액공제(제56조의3)의 주요 요건은 다음과 같습니다:

1. 전자계산서 발급 의무 대상자일 것
- 부가가치세법상 전자계산서 발급 의무 대상자에 해당해야 합니다.

2. 전자계산서를 정상적으로 발급할 것
- 전자계산서 발급 시스템을 구축하고 전자계산서를 정상적으로 발급해야 합니다.

3. 전자계산서 발급 내역을 성실하게 신고할 것
- 전자계산서 발급 내역을 부가가치세 신고 시 성실하게 신고해야 합니다.

4. 전자계산서 발급 금액이 일정 기준 이상일 것
- 전자계산서 발급 금액이 연간 3천만원 이상이어야 합니다.

5. 전자계산서 발급 시스템을 1년 이상 운영할 것
- 전자계산서 발급 시스템을 1년 이상 운영해야 합니다.

이상의 요건을 모두 충족해야 전자계산서 발급 세액공제를 받을 수 있습니다.


In [26]:
query = '연봉이 5000만원인 직장인의 산출세액을 알려주세요. 단, 현행 세법은 누진세율임에 유의하시오.'
answer = qa_chain.invoke(query)
print(answer)

# 하아..

연봉 5000만원인 직장인의 산출세액은 다음과 같습니다.

현행 소득세법에 따르면 연봉 5000만원은 종합소득세 과세표준 구간 중 가장 높은 구간에 해당합니다. 

과세표준 구간별 세율은 다음과 같습니다:
- 1200만원 이하: 6%
- 1200만원 초과 4600만원 이하: 15% 
- 4600만원 초과 8800만원 이하: 24%
- 8800만원 초과: 35%

연봉 5000만원의 경우, 산출세액은 다음과 같이 계산됩니다:
- 1200만원까지: 1200만원 × 6% = 72만원
- 1200만원 초과 4600만원까지: (4600만원 - 1200만원) × 15% = 510만원 
- 4600만원 초과 5000만원까지: (5000만원 - 4600만원) × 24% = 96만원

따라서 총 산출세액은 72만원 + 510만원 + 96만원 = 678만원입니다.
